# Comparación RAG: Qwen base y adaptadores LoRA entrenados con distintos prompts

Preparado para el repositorio `Chechu13/Practica_PLN_II`.

**Qué hace**
- Evalúa Qwen2.5-7B-Instruct sin fine-tuning y los adaptadores `qwen_lora_zero`, `qwen_lora_one` y `qwen_lora_few`.
- Usa la partición de entrenamiento/test del proyecto (`test_size=0.2`, `random_state=42`).
- Recupera los contextos de las preguntas de test usando `ContextRetriever` e `models/context_embeddings.pt` (solo recuperación local; no web/fallback).
- Reutiliza los mismos contextos recuperados para todas las variantes, para que la comparación del generador sea justa.
- Cada 10 respuestas imprime la pregunta, el contexto recuperado, la respuesta esperada y la generada, además de métricas y tiempos acumulados.
- Guarda cada predicción inmediatamente en JSONL y archivos parciales, de modo que no se pierda todo el trabajo si se interrumpe la ejecución.
- Utiliza las métricas del proyecto en `src/evaluate.py`: EM, F1 token-level, BLEU-1 y ROUGE-1.
- Reutiliza los CSV completos por modelo; genera solo las variantes sin resultados completos y reanuda CSV parciales.
- Guarda tanto CSV individuales por modelo como `all_predictions.csv` consolidado.
- No considera el tiempo de entrenamiento. El tiempo de carga se registra como dato secundario.



In [1]:
# 1. Configuración

from pathlib import Path
import sys, os, json, time, gc, random
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (
        (PROJECT_ROOT / "src" / "preprocess" / "preprocess.py").exists()
        and (PROJECT_ROOT / "scripts" / "experiments" / "train_qwen_prompt.py").exists()
    ):
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src" / "preprocess" / "preprocess.py").exists():
    raise FileNotFoundError(
        "No encuentro src/preprocess/preprocess.py. "
        "Abre Jupyter desde la raíz de Practica_PLN_II."
    )
sys.path.insert(0, str(PROJECT_ROOT))

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
SEED = 42
MAX_LENGTH = 384
MAX_NEW_TOKENS = 32
PRINT_EVERY = 10
WARMUP_QUESTIONS = 1
LIMIT_TEST = None  # Pon 5 para una prueba corta; None para todo el test
RUN_BASE_MODEL = True

INDEX_PATH = PROJECT_ROOT / "models" / "context_embeddings.pt"
QUESTIONS_PATH = PROJECT_ROOT / "data" / "questions_train.json"
CONTEXTS_PATH = PROJECT_ROOT / "data" / "contexts.json"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
OUTPUT_DIR = PROJECT_ROOT / "results" / "rag_prompt_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADAPTERS = {
    "FT_zero": PROJECT_ROOT / "models" / "prompt_training" / "qwen_lora_zero",
    "FT_one": PROJECT_ROOT / "models" / "prompt_training" / "qwen_lora_one",
    "FT_few": PROJECT_ROOT / "models" / "prompt_training" / "qwen_lora_few",
}
PROMPT_FILES = {"zero": "primer.md", "one": "one_shot.md", "few": "few_shot.md"}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DTYPE = torch.float32

print("Raíz:", PROJECT_ROOT)
print("Dispositivo:", DEVICE)
print("Precisión:", DTYPE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
for p in [QUESTIONS_PATH, CONTEXTS_PATH, INDEX_PATH]:
    print(("OK" if p.exists() else "FALTA"), p)


Raíz: C:\Users\chech\Practica_PLN_II
Dispositivo: cuda
Precisión: torch.bfloat16
GPU: NVIDIA GeForce RTX 5060 Ti
OK C:\Users\chech\Practica_PLN_II\data\questions_train.json
OK C:\Users\chech\Practica_PLN_II\data\contexts.json
OK C:\Users\chech\Practica_PLN_II\models\context_embeddings.pt


## 2. Cargar datos y reservar el test

Este bloque replica el `train_test_split` del script de entrenamiento. El contexto vinculado originalmente en `questions_train.json` se usa para construir los ejemplos del dataset, pero **se reemplaza en el test por el contexto que devuelva el recuperador**.

In [ ]:
from sklearn.model_selection import train_test_split
from src.preprocess.preprocess import prepare_dataset
from src.evaluate import exact_match_score, f1_score, bleu_score, rouge_score
from src.retrieval.context_retriever import ContextRetriever

for path in (QUESTIONS_PATH, CONTEXTS_PATH, INDEX_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Falta archivo requerido: {path}")

dataset = prepare_dataset(str(QUESTIONS_PATH), str(CONTEXTS_PATH))
train_data, test_data = train_test_split(
    dataset, test_size=0.2, random_state=SEED
)
train_data = [dict(x) for x in train_data]
test_data = [dict(x) for x in test_data]
if LIMIT_TEST is not None:
    test_data = test_data[:LIMIT_TEST]

print(f"Total: {len(dataset)} | Train: {len(train_data)} | Test: {len(test_data)}")
print("Respondibles en test:", sum(bool(x["has_answer"]) for x in test_data))
print("No respondibles en test:", sum(not bool(x["has_answer"]) for x in test_data))
display(pd.DataFrame(test_data)[["question", "answer", "has_answer"]].head())


## 3. Recuperar los contextos de test con el RAG local

Se utiliza el `ContextRetriever` que ya existe en el proyecto. Su método `retrieve()` devuelve el contexto local más relevante. Se recupera una sola vez y el mismo contexto se entrega a BASE y a todos los adaptadores. La búsqueda web no se utiliza, para conservar las condiciones del entrenamiento.

In [3]:
retriever = ContextRetriever(str(INDEX_PATH), device=DEVICE)

retrieved_test = []
retrieval_all_start = time.perf_counter()

for i, item in enumerate(test_data, start=1):
    retrieval_start = time.perf_counter()
    retrieved = retriever.retrieve(item["question"])
    retrieval_seconds = time.perf_counter() - retrieval_start

    record = {
        **item,
        "context": retrieved.get("context", "") or "",
        "retrieved_context_id": retrieved.get("context_id", "none"),
        "retrieval_score": float(retrieved.get("score", 0.0)),
        "retrieval_seconds": float(retrieval_seconds),
    }
    retrieved_test.append(record)

    if i % PRINT_EVERY == 0 or i == len(test_data):
        print(
            f"[Retrieval] {i}/{len(test_data)} | "
            f"score último={record['retrieval_score']:.3f} | "
            f"latencia último={retrieval_seconds:.2f}s",
            flush=True,
        )

retrieval_total_seconds = time.perf_counter() - retrieval_all_start
retrieval_df = pd.DataFrame([
    {
        "index": i,
        "question": x["question"],
        "reference": x["answer"] or "",
        "has_answer": bool(x["has_answer"]),
        "retrieved_context_id": x["retrieved_context_id"],
        "retrieval_score": x["retrieval_score"],
        "retrieval_seconds": x["retrieval_seconds"],
        "retrieved_context": x["context"],
    }
    for i, x in enumerate(retrieved_test)
])
retrieval_csv = OUTPUT_DIR / "retrieved_test_contexts.csv"
retrieval_df.to_csv(retrieval_csv, index=False)
print(f"Recuperación terminada: {retrieval_total_seconds:.1f}s")
print(f"Contextos guardados en: {retrieval_csv}")
print("Contextos vacíos:", int((retrieval_df["retrieved_context"].fillna("").str.strip() == "").sum()), "/", len(retrieval_df))
display(retrieval_df[["question", "retrieved_context_id", "retrieval_score", "retrieved_context"]].head(3))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Retrieval] 10/70 | score último=0.720 | latencia último=0.25s
[Retrieval] 20/70 | score último=0.694 | latencia último=0.24s
[Retrieval] 30/70 | score último=0.406 | latencia último=0.24s
[Retrieval] 40/70 | score último=0.493 | latencia último=0.23s
[Retrieval] 50/70 | score último=0.709 | latencia último=0.24s
[Retrieval] 60/70 | score último=0.599 | latencia último=0.24s
[Retrieval] 70/70 | score último=1.008 | latencia último=0.24s
Recuperación terminada: 17.3s
Contextos guardados en: C:\Users\chech\Practica_PLN_II\results\rag_prompt_comparison\retrieved_test_contexts.csv
Contextos vacíos: 0 / 70


,question,retrieved_context_id,retrieval_score,retrieved_context
0,Who established the progressive-education move...,c14764,0.604078,"In time, some ideas from these experiments and..."
1,By what other term is anti-aircraft warfare kn...,c06574,0.685762,Anti-aircraft warfare or counter-air defence i...
2,Which two countries are connected by the Vikin...,c02577,0.395512,SURE provide the telecommunications service in...


## 4. Preparar prompts y ejemplos

Se reutilizan los archivos de prompt del proyecto. Para one-shot y few-shot, los ejemplos se escogen únicamente del conjunto de entrenamiento y reciben contextos recuperados localmente, como durante el entrenamiento. No se usan preguntas del test como demostraciones.

In [4]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def retrieve_train_for_demos(samples, retriever):
    out = []
    for i, item in enumerate(samples, start=1):
        res = retriever.retrieve(item["question"])
        out.append({
            **item,
            "context": res.get("context", "") or "",
            "context_id": res.get("context_id", "none"),
            "retrieval_score": float(res.get("score", 0.0)),
        })
        if i % 50 == 0 or i == len(samples):
            print(f"[Train demos] contextos: {i}/{len(samples)}", flush=True)
    return out

def select_examples(train_samples):
    answerable = [
        x for x in train_samples
        if x.get("has_answer") and x.get("answer") and x.get("context")
    ]
    unanswerable = [x for x in train_samples if not x.get("has_answer")]
    if len(answerable) < 3:
        raise ValueError("No hay suficientes ejemplos respondibles en train.")
    short = min(answerable, key=lambda x: len(x["answer"].split()))
    medium_candidates = [
        x for x in answerable
        if x is not short and 1 <= len(x["answer"].split()) <= 5
    ]
    medium = medium_candidates[0] if medium_candidates else next(x for x in answerable if x is not short)
    remaining = [x for x in answerable if x is not short and x is not medium]
    long_answer = max(remaining, key=lambda x: len(x["answer"].split()))
    examples = [short, medium, long_answer]
    if unanswerable:
        examples.append(unanswerable[0])
    return [
        {
            "question": x["question"],
            "context": x.get("context", "") or "",
            "answer": x.get("answer") or "",
        }
        for x in examples[:4]
    ]

def build_system_prompt(strategy, examples):
    filename = PROMPT_FILES[strategy]
    path = PROMPTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"No encuentro el prompt: {path}")
    template = path.read_text(encoding="utf-8").strip()

    if strategy == "zero":
        return template
    if strategy == "one":
        if not examples:
            raise ValueError("One-shot necesita al menos un ejemplo de train.")
        e = examples[0]
        return template.format(
            example_context=e["context"],
            example_question=e["question"],
            example_answer=e["answer"],
        )
    if strategy == "few":
        if len(examples) < 4:
            raise ValueError("Few-shot necesita cuatro ejemplos para llenar la plantilla.")
        values = {}
        for i, e in enumerate(examples[:4], start=1):
            values[f"example_{i}_context"] = e["context"]
            values[f"example_{i}_question"] = e["question"]
            values[f"example_{i}_answer"] = e["answer"]
        return template.format(**values)
    raise ValueError(f"Estrategia desconocida: {strategy}")

# Reproducir el flujo del entrenamiento: recuperar contexto local de los ejemplos de train.
retrieved_train = retrieve_train_for_demos(train_data, retriever)
selected_examples = select_examples(retrieved_train)

examples_by_strategy = {
    "zero": [],
    "one": selected_examples[:1],
    "few": selected_examples[:4],
}
system_prompts = {
    strategy: build_system_prompt(strategy, examples)
    for strategy, examples in examples_by_strategy.items()
}
for name, prompt in system_prompts.items():
    print(f"{name}: {len(prompt)} caracteres; {len(examples_by_strategy[name])} ejemplos")


[Train demos] contextos: 50/280
[Train demos] contextos: 100/280
[Train demos] contextos: 150/280
[Train demos] contextos: 200/280
[Train demos] contextos: 250/280
[Train demos] contextos: 280/280
zero: 147 caracteres; 0 ejemplos
one: 1152 caracteres; 1 ejemplos
few: 4087 caracteres; 4 ejemplos


## 5. Funciones de generación, métricas y guardado incremental

La plantilla de generación usa `tokenizer.apply_chat_template`, igual que el script existente. La latencia de generación se mide separada de la de recuperación. Se guardan las predicciones en JSONL, CSV y JSON parcial tras cada pregunta.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.truncation_side = "left"
    return tokenizer

def load_model(adapter_path=None):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
    ).to(DEVICE)
    if adapter_path is not None:
        adapter_path = Path(adapter_path)
        if not (adapter_path / "adapter_config.json").exists():
            raise FileNotFoundError(f"No encuentro adapter_config.json en {adapter_path}")
        model = PeftModel.from_pretrained(model, str(adapter_path))
    model.eval()
    return model

def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.inference_mode()
def generate_one(model, tokenizer, item, system_prompt):
    context = item.get("context") or "No context is available."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {item['question']}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    sync_device()
    t0 = time.perf_counter()
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    sync_device()
    elapsed = time.perf_counter() - t0
    generated_ids = outputs[:, inputs["input_ids"].shape[1]:]
    answer = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    n_tokens = int(generated_ids.shape[1])
    return answer, elapsed, n_tokens, int(inputs["input_ids"].shape[1])

def summarize(records):
    ok = [x for x in records if not x.get("error")]
    ans = [x for x in ok if x["has_answer"]]
    unans = [x for x in ok if not x["has_answer"]]

    def mean(items, key):
        return float(np.mean([x[key] for x in items])) if items else float("nan")

    gens = [x["generation_seconds"] for x in ok]
    end_to_end = [x["total_seconds"] for x in ok]
    return {
        "n_total": len(records),
        "n_success": len(ok),
        "n_errors": sum(bool(x.get("error")) for x in records),
        "EM_percent": 100 * mean(ok, "em"),
        "F1_percent": 100 * mean(ok, "f1"),
        "answerable_F1_percent": 100 * mean(ans, "f1"),
        "unanswerable_F1_percent": 100 * mean(unans, "f1"),
        "empty_prediction_rate_percent": 100 * mean(ok, "empty_prediction"),
        "generation_total_seconds": float(np.sum(gens)) if gens else float("nan"),
        "generation_mean_seconds": float(np.mean(gens)) if gens else float("nan"),
        "generation_p50_seconds": float(np.percentile(gens, 50)) if gens else float("nan"),
        "generation_p95_seconds": float(np.percentile(gens, 95)) if gens else float("nan"),
        "rag_total_mean_seconds": float(np.mean(end_to_end)) if end_to_end else float("nan"),
        "mean_tokens_per_second": mean(ok, "tokens_per_second"),
        "mean_generated_tokens": mean(ok, "generated_tokens"),
    }

def save_record(variant, record, records):
    jsonl_path = OUTPUT_DIR / f"{variant.lower()}_predictions.jsonl"
    with open(jsonl_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
    pd.DataFrame(records).to_csv(
        OUTPUT_DIR / f"{variant.lower()}_predictions_partial.csv",
        index=False
    )
    with open(OUTPUT_DIR / f"{variant.lower()}_metrics_partial.json", "w", encoding="utf-8") as f:
        json.dump(summarize(records), f, ensure_ascii=False, indent=2)

print("Funciones listas.")


Funciones listas.


## 6. Ejecutar evaluación

Cada adaptador se evalúa con su prompt de entrenamiento correspondiente. `BASE` usa zero-shot como referencia. Para hacer la primera prueba rápida, pon `LIMIT_TEST = 5` arriba y, si funciona, vuelve a `None` para las 70 preguntas.

In [ ]:
# Reutilizar resultados por modelo y ejecutar solo los que falten.
from src.evaluate import bleu_score, rouge_score

set_seed()
variants = []
if RUN_BASE_MODEL:
    variants.append(("BASE", None, "zero"))
variants.extend([
    ("FT_zero", ADAPTERS["FT_zero"], "zero"),
    ("FT_one", ADAPTERS["FT_one"], "one"),
    ("FT_few", ADAPTERS["FT_few"], "few"),
])

expected_n = len(retrieved_test)
summaries = []
all_records = []
pending_variants = []

def model_csv_candidates(variant_name):
    # Preferir CSV final por modelo si existe; usar el parcial si no.
    return [
        OUTPUT_DIR / f"{variant_name.lower()}_predictions.csv",
        OUTPUT_DIR / f"{variant_name.lower()}_predictions_partial.csv",
    ]

def load_model_records(variant_name):
    for path in model_csv_candidates(variant_name):
        if not path.exists():
            continue
        try:
            cached = pd.read_csv(path)
            if "variant" not in cached.columns:
                cached["variant"] = variant_name
            cached = cached[cached["variant"] == variant_name].copy()
            if "index" not in cached.columns:
                continue
            cached["index"] = pd.to_numeric(cached["index"], errors="coerce")
            cached = cached.dropna(subset=["index"])
            cached["index"] = cached["index"].astype(int)
            cached = cached.drop_duplicates(subset=["index"], keep="last")
            cached = cached.sort_values("index")
            # Solo reutilizar como completo si están todas las filas esperadas,
            # los índices son 0..N-1 y no hay errores de generación.
            complete = (
                len(cached) == expected_n
                and cached["index"].tolist() == list(range(expected_n))
                and ("error" not in cached.columns or cached["error"].isna().all())
                and {"prediction", "reference", "question"}.issubset(cached.columns)
            )
            if complete:
                print(f"[CACHÉ] {variant_name}: {len(cached)}/{expected_n} predicciones desde {path.name}")
                return cached, True
            print(f"[CACHÉ PARCIAL] {variant_name}: {len(cached)}/{expected_n}; se completarán las preguntas pendientes.")
            return cached, False
        except Exception as exc:
            print(f"[CACHÉ] No se pudo leer {path.name}: {exc}")
    return pd.DataFrame(), False

# Primero inspeccionar los CSV individuales. No se carga ningún modelo aquí.
cache_by_variant = {}
complete_by_variant = {}
for variant_name, adapter_path, strategy in variants:
    cached, complete = load_model_records(variant_name)
    cache_by_variant[variant_name] = cached
    complete_by_variant[variant_name] = complete
    if not complete:
        pending_variants.append((variant_name, adapter_path, strategy))

# Solo cargar tokenizer/modelos si al menos una variante necesita generación.
tokenizer = load_tokenizer() if pending_variants else None

for variant_name, adapter_path, strategy in variants:
    print("\\n" + "=" * 84)
    print(f"PROCESANDO {variant_name} | prompt={strategy}")
    print("=" * 84, flush=True)

    cached = cache_by_variant[variant_name]
    if complete_by_variant[variant_name]:
        records = cached.to_dict("records")
        # Recalcular métricas de texto sobre el CSV sin ejecutar inferencia.
        for record in records:
            pred = "" if pd.isna(record.get("prediction")) else str(record.get("prediction", ""))
            ref = "" if pd.isna(record.get("reference")) else str(record.get("reference", ""))
            record["BLEU_percent"] = 100 * bleu_score(pred, ref)
            record["ROUGE_percent"] = 100 * rouge_score(pred, ref)
        summary = summarize(records)
        summary.update({
            "variant": variant_name,
            "prompt_strategy": strategy,
            "adapter_path": str(adapter_path) if adapter_path else "BASE",
            "model_load_seconds_secondary": (
                float(cached["model_load_seconds_secondary"].dropna().iloc[0])
                if "model_load_seconds_secondary" in cached.columns and cached["model_load_seconds_secondary"].notna().any()
                else float("nan")
            ),
            "peak_vram_gb": (
                float(cached["peak_vram_gb"].dropna().iloc[0])
                if "peak_vram_gb" in cached.columns and cached["peak_vram_gb"].notna().any()
                else float("nan")
            ),
            "variant_wall_seconds": float(pd.to_numeric(cached.get("variant_wall_seconds", pd.Series(dtype=float)), errors="coerce").dropna().iloc[0])
                if "variant_wall_seconds" in cached.columns and pd.to_numeric(cached["variant_wall_seconds"], errors="coerce").notna().any()
                else float("nan"),
            "BLEU_percent": float(np.mean([r["BLEU_percent"] for r in records])) if records else float("nan"),
            "ROUGE_percent": float(np.mean([r["ROUGE_percent"] for r in records])) if records else float("nan"),
        })
        summaries.append(summary)
        all_records.extend(records)
        continue

    if adapter_path is not None and not Path(adapter_path).exists():
        print(f"OMITIDO: no existe {adapter_path}", flush=True)
        continue

    # Reanudar desde las filas válidas del CSV parcial, si las hay.
    existing_by_index = {
        int(row["index"]): row
        for row in cached.to_dict("records")
        if pd.notna(row.get("index"))
        and (not row.get("error") or pd.isna(row.get("error")))
    }
    records_by_index = dict(existing_by_index)
    missing_items = [
        (i, item) for i, item in enumerate(retrieved_test)
        if i not in records_by_index
    ]
    print(f"{variant_name}: {len(records_by_index)} existentes; {len(missing_items)} por generar.")

    if not missing_items:
        records = [records_by_index[i] for i in sorted(records_by_index)]
        for record in records:
            pred = "" if pd.isna(record.get("prediction")) else str(record.get("prediction", ""))
            ref = "" if pd.isna(record.get("reference")) else str(record.get("reference", ""))
            record["BLEU_percent"] = 100 * bleu_score(pred, ref)
            record["ROUGE_percent"] = 100 * rouge_score(pred, ref)
        summary = summarize(records)
        summary.update({"variant": variant_name, "prompt_strategy": strategy,
                        "adapter_path": str(adapter_path) if adapter_path else "BASE"})
        summary["BLEU_percent"] = float(np.mean([r["BLEU_percent"] for r in records]))
        summary["ROUGE_percent"] = float(np.mean([r["ROUGE_percent"] for r in records]))
        summaries.append(summary)
        all_records.extend(records)
        continue

    load_start = time.perf_counter()
    try:
        model = load_model(adapter_path)
    except Exception as exc:
        print(f"ERROR al cargar {variant_name}: {exc}", flush=True)
        summaries.append({"variant": variant_name, "load_error": repr(exc)})
        continue
    load_seconds = time.perf_counter() - load_start
    print(f"Modelo cargado en {load_seconds:.1f}s", flush=True)

    for j in range(min(WARMUP_QUESTIONS, len(retrieved_test))):
        _ = generate_one(model, tokenizer, retrieved_test[j], system_prompts[strategy])

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    jsonl_path = OUTPUT_DIR / f"{variant_name.lower()}_predictions.jsonl"
    # Crear JSONL vacío solo si no había datos previos para reanudar.
    if not jsonl_path.exists() or not records_by_index:
        jsonl_path.write_text("", encoding="utf-8")

    variant_start = time.perf_counter()
    for i, item in missing_items:
        try:
            prediction, gen_seconds, n_tokens, input_tokens = generate_one(
                model, tokenizer, item, system_prompts[strategy]
            )
            record = {
                "variant": variant_name,
                "prompt_strategy": strategy,
                "index": i,
                "question": item["question"],
                "reference": item.get("answer") or "",
                "prediction": prediction,
                "has_answer": bool(item["has_answer"]),
                "retrieved_context_id": item["retrieved_context_id"],
                "retrieval_score": item["retrieval_score"],
                "retrieved_context": item["context"],
                "retrieval_seconds": float(item["retrieval_seconds"]),
                "generation_seconds": float(gen_seconds),
                "total_seconds": float(item["retrieval_seconds"] + gen_seconds),
                "generated_tokens": int(n_tokens),
                "input_tokens": int(input_tokens),
                "tokens_per_second": n_tokens / gen_seconds if gen_seconds > 0 else float("nan"),
                "em": float(exact_match_score(prediction, item.get("answer") or "")),
                "f1": float(f1_score(prediction, item.get("answer") or "")),
                "BLEU_percent": 100 * bleu_score(prediction, item.get("answer") or ""),
                "ROUGE_percent": 100 * rouge_score(prediction, item.get("answer") or ""),
                "empty_prediction": not bool(prediction.strip()),
                "error": None,
            }
        except Exception as exc:
            record = {
                "variant": variant_name, "prompt_strategy": strategy, "index": i,
                "question": item["question"], "reference": item.get("answer") or "",
                "prediction": "", "has_answer": bool(item["has_answer"]),
                "retrieved_context_id": item["retrieved_context_id"],
                "retrieval_score": item["retrieval_score"], "retrieved_context": item["context"],
                "retrieval_seconds": float(item["retrieval_seconds"]),
                "generation_seconds": float("nan"), "total_seconds": float("nan"),
                "generated_tokens": 0, "input_tokens": 0, "tokens_per_second": float("nan"),
                "em": 0.0, "f1": 0.0, "BLEU_percent": 0.0, "ROUGE_percent": 0.0,
                "empty_prediction": True, "error": repr(exc),
            }
            print(f"ERROR {variant_name} pregunta {i + 1}: {exc}", flush=True)

        records_by_index[i] = record
        current_records = [records_by_index[k] for k in sorted(records_by_index)]
        # Mantener el CSV individual actualizado tras cada pregunta.
        pd.DataFrame(current_records).to_csv(
            OUTPUT_DIR / f"{variant_name.lower()}_predictions_partial.csv",
            index=False, encoding="utf-8-sig"
        )
        with open(jsonl_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\\n")
            f.flush()
            os.fsync(f.fileno())

        if len(records_by_index) % PRINT_EVERY == 0 or len(records_by_index) == expected_n:
            current = summarize(current_records)
            print(
                f"[{variant_name}] {len(records_by_index)}/{expected_n} | "
                f"EM={current['EM_percent']:.2f}% | F1={current['F1_percent']:.2f}%",
                flush=True,
            )

    records = [records_by_index[k] for k in sorted(records_by_index)]
    peak_vram_gb = (
        torch.cuda.max_memory_allocated() / (1024**3)
        if torch.cuda.is_available() else float("nan")
    )
    final_summary = summarize(records)
    final_summary.update({
        "variant": variant_name,
        "prompt_strategy": strategy,
        "adapter_path": str(adapter_path) if adapter_path else "BASE",
        "model_load_seconds_secondary": float(load_seconds),
        "peak_vram_gb": float(peak_vram_gb),
        "variant_wall_seconds": float(time.perf_counter() - variant_start),
        "BLEU_percent": float(np.mean([r["BLEU_percent"] for r in records])),
        "ROUGE_percent": float(np.mean([r["ROUGE_percent"] for r in records])),
    })
    summaries.append(final_summary)
    all_records.extend(records)

    pd.DataFrame(summaries).to_csv(OUTPUT_DIR / "summary_metrics_partial.csv", index=False)
    pd.DataFrame(all_records).to_csv(OUTPUT_DIR / "all_predictions_partial.csv", index=False)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Fin de {variant_name}.", flush=True)

summary_df = pd.DataFrame(summaries)
predictions_df = pd.DataFrame(all_records)
if not predictions_df.empty:
    predictions_df = predictions_df.sort_values(["variant", "index"]).reset_index(drop=True)
display(summary_df)


## 7. Resultados finales

El tiempo de generación no incluye recuperación ni carga del modelo. `rag_total_mean_seconds` estima el tiempo por pregunta como recuperación + generación. La recuperación se ha ejecutado una vez para todos los modelos; la latencia por modelo se presenta por separado para no contabilizar varias veces la misma recuperación.

In [ ]:
summary_path = OUTPUT_DIR / "summary_metrics.csv"
predictions_path = OUTPUT_DIR / "all_predictions.csv"

# Guardar un CSV final por cada modelo, además del CSV conjunto.
for model_name in predictions_df["variant"].dropna().unique():
    model_df = predictions_df[predictions_df["variant"] == model_name].copy()
    model_df.to_csv(
        OUTPUT_DIR / f"{model_name.lower()}_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )

summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
predictions_df.to_csv(predictions_path, index=False, encoding="utf-8-sig")

print("Resumen final:", summary_path)
print("Predicciones conjuntas:", predictions_path)
print("CSV individuales:", ", ".join(
    f"{name.lower()}_predictions.csv"
    for name in predictions_df["variant"].dropna().unique()
))

if not summary_df.empty and {"F1_percent", "generation_mean_seconds"}.issubset(summary_df.columns):
    import matplotlib.pyplot as plt
    plot_df = summary_df.dropna(subset=["F1_percent", "generation_mean_seconds"])
    if len(plot_df):
        plt.figure()
        plt.scatter(plot_df["generation_mean_seconds"], plot_df["F1_percent"])
        for _, row in plot_df.iterrows():
            plt.annotate(row["variant"], (row["generation_mean_seconds"], row["F1_percent"]))
        plt.xlabel("Latencia media de generación (s/pregunta)")
        plt.ylabel("F1 (%)")
        plt.title("Calidad frente a velocidad")
        plt.grid(True)
        plt.show()


In [ ]:
quality_cols = [
    "variant",
    "n_total",
    "EM_percent",
    "F1_percent",
    "BLEU_percent",
    "ROUGE_percent",
    "answerable_F1_percent",
    "unanswerable_F1_percent",
]

quality_table = (
    summary_df.loc[
        summary_df["variant"].isin(["BASE", "FT_zero", "FT_one", "FT_few"]),
        quality_cols,
    ]
    .rename(columns={
        "variant": "Modelo",
        "n_total": "Preguntas",
        "EM_percent": "EM (%)",
        "F1_percent": "F1 (%)",
        "BLEU_percent": "BLEU-1 (%)",
        "ROUGE_percent": "ROUGE-1 (%)",
        "answerable_F1_percent": "F1 respondibles (%)",
        "unanswerable_F1_percent": "F1 no respondibles (%)",
    })
    .round(2)
    .reset_index(drop=True)
)

display(quality_table)

In [9]:
efficiency_table = (
    summary_df.loc[
        summary_df["variant"].isin(["BASE", "FT_zero", "FT_one", "FT_few"]),
        [
            "variant",
            "F1_percent",
            "generation_mean_seconds",
            "generation_p95_seconds",
            "mean_tokens_per_second",
            "peak_vram_gb",
        ],
    ]
    .rename(columns={
        "variant": "Modelo",
        "F1_percent": "F1 (%)",
        "generation_mean_seconds": "Tiempo medio (s)",
        "generation_p95_seconds": "P95 (s)",
        "mean_tokens_per_second": "Tokens/s",
        "peak_vram_gb": "VRAM máxima (GB)",
    })
    .round(2)
    .reset_index(drop=True)
)

display(efficiency_table)

,Modelo,F1 (%),Tiempo medio (s),P95 (s),Tokens/s,VRAM máxima (GB)
0,BASE,31.11,4.65,10.26,4.37,14.41
1,FT_zero,81.86,1.80,2.78,3.47,14.43
2,FT_one,67.28,0.42,0.76,11.53,14.43
3,FT_few,79.86,0.33,0.55,11.91,14.43


In [10]:
base_rows = summary_df.loc[summary_df["variant"] == "BASE"]

if base_rows.empty:
    print("No hay resultados de BASE para calcular las mejoras.")
else:
    base = base_rows.iloc[0]
    ft_table = summary_df[
        summary_df["variant"].isin(["FT_zero", "FT_one", "FT_few"])
    ].copy()

    ft_table["Δ EM (pp)"] = ft_table["EM_percent"] - base["EM_percent"]
    ft_table["Δ F1 (pp)"] = ft_table["F1_percent"] - base["F1_percent"]

    if base["F1_percent"] != 0:
        ft_table["Mejora relativa F1 (%)"] = (
            (ft_table["F1_percent"] - base["F1_percent"])
            / base["F1_percent"] * 100
        )
    else:
        ft_table["Mejora relativa F1 (%)"] = float("nan")

    improvement_table = (
        ft_table[
            ["variant", "Δ EM (pp)", "Δ F1 (pp)", "Mejora relativa F1 (%)"]
        ]
        .rename(columns={"variant": "Modelo"})
        .round(2)
        .reset_index(drop=True)
    )

    display(improvement_table)

,Modelo,Δ EM (pp),Δ F1 (pp),Mejora relativa F1 (%)
0,FT_zero,62.86,50.75,163.10
1,FT_one,42.86,36.17,116.25
2,FT_few,60.00,48.75,156.67


In [13]:
error_rows = []

for model_name in ["BASE", "FT_zero", "FT_one", "FT_few"]:
    model_df = predictions_df[
        (predictions_df["variant"] == model_name)
        & predictions_df["error"].isna()
    ]

    answerable = model_df[model_df["has_answer"] == True]
    unanswerable = model_df[model_df["has_answer"] == False]

    error_rows.append({
        "Modelo": model_name,
        "Preguntas respondibles": len(answerable),
        "Respondibles con respuesta vacía": int(
            answerable["empty_prediction"].sum()
        ),
        "Preguntas no respondibles": len(unanswerable),
        "No respondibles con respuesta vacía": int(
            unanswerable["empty_prediction"].sum()
        ),
        "Total de respuestas vacías": int(
            model_df["empty_prediction"].sum()
        ),
        "Errores de ejecución": int(
            (
                predictions_df["variant"].eq(model_name)
                & predictions_df["error"].notna()
            ).sum()
        ),
    })

error_table = pd.DataFrame(error_rows)
display(error_table)

,Modelo,Preguntas respondibles,Respondibles con respuesta vacía,Preguntas no respondibles,No respondibles con respuesta vacía,Total de respuestas vacías,Errores de ejecución
0,BASE,54,0,16,0,0,0
1,FT_zero,54,0,16,15,15,0
2,FT_one,54,0,16,12,12,0
3,FT_few,54,0,16,15,15,0


In [11]:
strategies = [
    ("FT_zero", "FT_one", "FT_zero vs FT_one"),
    ("FT_zero", "FT_few", "FT_zero vs FT_few"),
    ("FT_one", "FT_few", "FT_one vs FT_few"),
]

strategy_rows = []

for model_a, model_b, comparison in strategies:
    rows_a = summary_df.loc[summary_df["variant"] == model_a]
    rows_b = summary_df.loc[summary_df["variant"] == model_b]

    if rows_a.empty or rows_b.empty:
        continue

    a = rows_a.iloc[0]
    b = rows_b.iloc[0]

    strategy_rows.append({
        "Comparación": comparison,
        "Δ F1 (pp)": a["F1_percent"] - b["F1_percent"],
        "Δ tiempo medio (s)": (
            a["generation_mean_seconds"] - b["generation_mean_seconds"]
        ),
        "Δ EM (pp)": a["EM_percent"] - b["EM_percent"],
    })

strategy_table = pd.DataFrame(strategy_rows).round(2)
display(strategy_table)

,Comparación,Δ F1 (pp),Δ tiempo medio (s),Δ EM (pp)
0,FT_zero vs FT_one,14.58,1.38,20.00
1,FT_zero vs FT_few,2.00,1.47,2.86
2,FT_one vs FT_few,-12.58,0.09,-17.14


In [14]:
tables = {
    "quality_table": quality_table,
    "efficiency_table": efficiency_table,
    "improvement_table": improvement_table,
    "error_table": error_table,
    "strategy_table": strategy_table,
}

for name, table in tables.items():
    path = OUTPUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Guardado: {path}")

Guardado: C:\Users\chech\Practica_PLN_II\results\rag_prompt_comparison\quality_table.csv
Guardado: C:\Users\chech\Practica_PLN_II\results\rag_prompt_comparison\efficiency_table.csv
Guardado: C:\Users\chech\Practica_PLN_II\results\rag_prompt_comparison\improvement_table.csv
Guardado: C:\Users\chech\Practica_PLN_II\results\rag_prompt_comparison\error_table.csv
Guardado: C:\Users\chech\Practica_PLN_II\results\rag_prompt_comparison\strategy_table.csv
